# OULAD Complete Evaluation Pipeline

This notebook consolidates all three evaluation strategies:
1. **Random Split**: 5-fold stratified cross-validation
2. **LCPO (Leave-Course-Presentation-Out)**: Train on N-1 courses, test on 1
3. **Future-Presentation Split**: Train on earlier presentations, test on later ones

## Label Convention
- **1 = at-risk** (Fail/Withdrawn) - positive class requiring intervention
- **0 = success** (Pass/Distinction) - negative class

All metrics (precision, recall, F1, AUPRC) refer to identifying at-risk students.

## Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add src directory to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Import configuration
from config import (
    DATA_DIR, RESULTS_DIR, MODELS_DIR,
    LABEL_MAPPING, PREDICTION_WINDOWS,
    MODEL_PARAMS, RANDOM_STATE
)

# Machine learning imports
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print("✓ All imports successful")
print(f"Project root: {project_root}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Label mapping: {LABEL_MAPPING}")

## Load Existing Evaluation Scripts

We'll import the evaluation functions from our existing scripts.

In [ ]:
# Import evaluation scripts
import baseline_evaluation
import lcpo_evaluation
import future_presentation_evaluation

print("✓ Evaluation modules imported successfully")

## Data Loading

In [ ]:
def load_all_data():
    """Load all OULAD datasets"""
    print("Loading OULAD datasets...")
    
    student_info = pd.read_csv(DATA_DIR / "studentInfo.csv")
    student_vle = pd.read_csv(DATA_DIR / "studentVle.csv")
    student_assessment = pd.read_csv(DATA_DIR / "studentAssessment.csv")
    assessments = pd.read_csv(DATA_DIR / "assessments.csv")
    vle = pd.read_csv(DATA_DIR / "vle.csv")
    courses = pd.read_csv(DATA_DIR / "courses.csv")
    
    # Apply label mapping: 1=at-risk, 0=success
    student_info["target"] = student_info["final_result"].map(LABEL_MAPPING)
    
    print(f"✓ Loaded {len(student_info):,} students")
    print(f"  - At-risk (1): {(student_info['target'] == 1).sum():,} ({(student_info['target'] == 1).mean()*100:.1f}%)")
    print(f"  - Success (0): {(student_info['target'] == 0).sum():,} ({(student_info['target'] == 0).mean()*100:.1f}%)")
    
    # Display course presentation distribution
    print("\nCourse Presentation Distribution:")
    course_dist = student_info.groupby(['code_module', 'code_presentation']).size().reset_index(name='count')
    course_dist['course_pres'] = course_dist['code_module'] + '_' + course_dist['code_presentation']
    print(course_dist[['course_pres', 'count']].to_string(index=False))
    
    return {
        'student_info': student_info,
        'student_vle': student_vle,
        'student_assessment': student_assessment,
        'assessments': assessments,
        'vle': vle,
        'courses': courses
    }

# Load data
data = load_all_data()

---
# Evaluation 1: Random Split (5-Fold Cross-Validation)

Run the baseline evaluation with random stratified splits.

In [ ]:
print("="*80)
print("RUNNING RANDOM SPLIT EVALUATION")
print("="*80)
print("This will run 5-fold cross-validation for:")
print(f"  - Prediction weeks: {PREDICTION_WINDOWS}")
print(f"  - Feature groups: demographics, vle, assessment, all")
print(f"  - Models: Logistic Regression, Random Forest, XGBoost, LightGBM")
print("\nThis may take 10-15 minutes...\n")

# Run baseline evaluation using the imported module
# The baseline_evaluation module has a main() function that runs everything
import importlib
importlib.reload(baseline_evaluation)

# Or run it directly
!cd ../src && python baseline_evaluation.py

In [ ]:
# Load and display random split results
random_results_path = RESULTS_DIR / "baseline" / "baseline_results_detailed.csv"

if random_results_path.exists():
    random_results = pd.read_csv(random_results_path)
    print("✓ Random split results loaded")
    print(f"\nShape: {random_results.shape}")
    print(f"\nColumns: {list(random_results.columns)}")
    print("\nTop 10 Results by AUROC:")
    display(random_results.nlargest(10, 'AUROC')[['model', 'prediction_week', 'feature_group', 'AUROC', 'AUPRC', 'F1']])
else:
    print("⚠ Random split results not found. Please run the evaluation first.")
    random_results = None

---
# Evaluation 2: Leave-Course-Presentation-Out (LCPO)

Train on N-1 course presentations, test on the held-out presentation.

In [ ]:
print("="*80)
print("RUNNING LCPO EVALUATION")
print("="*80)
print("This will run leave-course-presentation-out evaluation for:")
print(f"  - Prediction weeks: {PREDICTION_WINDOWS}")
print(f"  - Feature groups: demographics, vle, assessment, all")
print(f"  - Models: Logistic Regression, Random Forest, XGBoost, LightGBM")
print("\nThis may take 15-20 minutes...\n")

# Run LCPO evaluation
!cd ../src && python lcpo_evaluation.py

In [ ]:
# Load and display LCPO results
lcpo_results_path = RESULTS_DIR / "lcpo" / "lcpo_results_detailed.csv"

if lcpo_results_path.exists():
    lcpo_results = pd.read_csv(lcpo_results_path)
    print("✓ LCPO results loaded")
    print(f"\nShape: {lcpo_results.shape}")
    print(f"\nColumns: {list(lcpo_results.columns)}")
    print("\nTop 10 Results by AUROC:")
    display(lcpo_results.nlargest(10, 'AUROC')[['model', 'prediction_week', 'feature_group', 'AUROC', 'AUPRC', 'F1']])
    
    # Show per-course variation
    print("\nCourse-Level Performance Variation (Week 16, All Features, LightGBM):")
    per_course_path = RESULTS_DIR / "lcpo" / "lcpo_per_course_week16_all_LightGBM.csv"
    if per_course_path.exists():
        per_course = pd.read_csv(per_course_path)
        display(per_course[['test_course', 'AUROC', 'AUPRC', 'F1', 'test_size']].sort_values('AUROC', ascending=False))
else:
    print("⚠ LCPO results not found. Please run the evaluation first.")
    lcpo_results = None

---
# Evaluation 3: Future-Presentation Split

Train on earlier presentations (2013B, 2013J, 2014B), test on future presentations (2014J).

In [ ]:
print("="*80)
print("RUNNING FUTURE-PRESENTATION EVALUATION")
print("="*80)
print("This will run temporal generalization evaluation:")
print("  - Train: 2013B, 2013J, 2014B")
print("  - Test: 2014J")
print(f"  - Prediction weeks: {PREDICTION_WINDOWS}")
print(f"  - Feature groups: demographics, vle, assessment, all")
print(f"  - Models: Logistic Regression, Random Forest, XGBoost, LightGBM")
print("\nThis may take 5-10 minutes...\n")

# Run future-presentation evaluation
!cd ../src && python future_presentation_evaluation.py

In [ ]:
# Load and display future-presentation results
future_results_path = RESULTS_DIR / "cross_course" / "future_presentation_results.csv"

if future_results_path.exists():
    future_results = pd.read_csv(future_results_path)
    print("✓ Future-presentation results loaded")
    print(f"\nShape: {future_results.shape}")
    print(f"\nColumns: {list(future_results.columns)}")
    print("\nTop 10 Results by AUROC:")
    display(future_results.nlargest(10, 'AUROC')[['model', 'prediction_week', 'feature_group', 'AUROC', 'AUPRC', 'F1']])
else:
    print("⚠ Future-presentation results not found. Please run the evaluation first.")
    future_results = None

---
# Comprehensive Results Comparison

In [ ]:
# Create comprehensive comparison table
if random_results is not None and lcpo_results is not None and future_results is not None:
    print("="*80)
    print("COMPREHENSIVE RESULTS COMPARISON")
    print("="*80)
    
    # Filter for week 16 and all features for comparison
    week = 16
    feature_group = 'all'
    
    comparison_data = []
    
    # Random split
    for model in ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']:
        row = random_results[
            (random_results['prediction_week'] == week) & 
            (random_results['feature_group'] == feature_group) &
            (random_results['model'] == model)
        ]
        if len(row) > 0:
            comparison_data.append({
                'Model': model,
                'Random_AUROC': row['AUROC'].values[0],
                'Random_AUPRC': row['AUPRC'].values[0],
                'Random_F1': row['F1'].values[0]
            })
    
    # LCPO
    for i, row_data in enumerate(comparison_data):
        model = row_data['Model']
        row = lcpo_results[
            (lcpo_results['prediction_week'] == week) & 
            (lcpo_results['feature_group'] == feature_group) &
            (lcpo_results['model'] == model)
        ]
        if len(row) > 0:
            comparison_data[i]['LCPO_AUROC'] = row['AUROC'].values[0]
            comparison_data[i]['LCPO_AUPRC'] = row['AUPRC'].values[0]
            comparison_data[i]['LCPO_F1'] = row['F1'].values[0]
    
    # Future-presentation
    for i, row_data in enumerate(comparison_data):
        model = row_data['Model']
        row = future_results[
            (future_results['prediction_week'] == week) & 
            (future_results['feature_group'] == feature_group) &
            (future_results['model'] == model)
        ]
        if len(row) > 0:
            comparison_data[i]['Future_AUROC'] = row['AUROC'].values[0]
            comparison_data[i]['Future_AUPRC'] = row['AUPRC'].values[0]
            comparison_data[i]['Future_F1'] = row['F1'].values[0]
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print(f"\nWeek {week} - All Features Comparison:")
    display(comparison_df)
    
    # Save comparison
    comparison_path = RESULTS_DIR / "comprehensive_comparison_week16.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print(f"\n✓ Comparison saved to {comparison_path}")
else:
    print("⚠ Not all results available for comparison")

---
# Visualizations

In [ ]:
# Visualization 1: AUROC comparison across evaluation types
if random_results is not None and lcpo_results is not None and future_results is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: AUROC by model and evaluation type (Week 16, All features)
    ax1 = axes[0]
    models = ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']
    x = np.arange(len(models))
    width = 0.25
    
    random_auroc = [comparison_df[comparison_df['Model'] == m]['Random_AUROC'].values[0] for m in models]
    lcpo_auroc = [comparison_df[comparison_df['Model'] == m]['LCPO_AUROC'].values[0] for m in models]
    future_auroc = [comparison_df[comparison_df['Model'] == m]['Future_AUROC'].values[0] for m in models]
    
    ax1.bar(x - width, random_auroc, width, label='Random Split', alpha=0.8)
    ax1.bar(x, lcpo_auroc, width, label='LCPO', alpha=0.8)
    ax1.bar(x + width, future_auroc, width, label='Future-Presentation', alpha=0.8)
    
    ax1.set_xlabel('Model', fontsize=12)
    ax1.set_ylabel('AUROC', fontsize=12)
    ax1.set_title('AUROC Comparison (Week 16, All Features)', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(models, rotation=45, ha='right')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim([0.7, 0.9])
    
    # Plot 2: AUPRC by model and evaluation type
    ax2 = axes[1]
    random_auprc = [comparison_df[comparison_df['Model'] == m]['Random_AUPRC'].values[0] for m in models]
    lcpo_auprc = [comparison_df[comparison_df['Model'] == m]['LCPO_AUPRC'].values[0] for m in models]
    future_auprc = [comparison_df[comparison_df['Model'] == m]['Future_AUPRC'].values[0] for m in models]
    
    ax2.bar(x - width, random_auprc, width, label='Random Split', alpha=0.8)
    ax2.bar(x, lcpo_auprc, width, label='LCPO', alpha=0.8)
    ax2.bar(x + width, future_auprc, width, label='Future-Presentation', alpha=0.8)
    
    ax2.set_xlabel('Model', fontsize=12)
    ax2.set_ylabel('AUPRC', fontsize=12)
    ax2.set_title('AUPRC Comparison (Week 16, All Features)', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(models, rotation=45, ha='right')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim([0.7, 0.9])
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'evaluation_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Visualization saved to results/evaluation_comparison.png")

In [ ]:
# Visualization 2: Feature group comparison
if random_results is not None:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Filter for LightGBM, Week 16
    lgbm_data = random_results[
        (random_results['model'] == 'LightGBM') & 
        (random_results['prediction_week'] == 16)
    ]
    
    feature_groups = ['demographics', 'vle', 'assessment', 'all']
    auroc_values = [lgbm_data[lgbm_data['feature_group'] == fg]['AUROC'].values[0] for fg in feature_groups]
    auprc_values = [lgbm_data[lgbm_data['feature_group'] == fg]['AUPRC'].values[0] for fg in feature_groups]
    
    x = np.arange(len(feature_groups))
    width = 0.35
    
    ax.bar(x - width/2, auroc_values, width, label='AUROC', alpha=0.8)
    ax.bar(x + width/2, auprc_values, width, label='AUPRC', alpha=0.8)
    
    ax.set_xlabel('Feature Group', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Feature Group Comparison (LightGBM, Week 16)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(feature_groups)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0.6, 0.9])
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'feature_group_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Visualization saved to results/feature_group_comparison.png")

---
# Summary Report

In [ ]:
print("="*80)
print("OULAD EVALUATION SUMMARY REPORT")
print("="*80)
print(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nLabel Convention: 1=at-risk (Fail/Withdrawn), 0=success (Pass/Distinction)")
print(f"\nPrediction Windows: {PREDICTION_WINDOWS}")
print(f"Models: Logistic Regression, Random Forest, XGBoost, LightGBM")
print(f"Feature Groups: demographics, vle, assessment, all")

if random_results is not None and lcpo_results is not None and future_results is not None:
    print("\n" + "="*80)
    print("BEST OVERALL RESULTS (Week 16, All Features)")
    print("="*80)
    
    for eval_type, df in [('Random Split', random_results), ('LCPO', lcpo_results), ('Future-Presentation', future_results)]:
        subset = df[(df['prediction_week'] == 16) & (df['feature_group'] == 'all')]
        best = subset.nlargest(1, 'AUROC').iloc[0]
        print(f"\n{eval_type}:")
        print(f"  Best Model: {best['model']}")
        print(f"  AUROC: {best['AUROC']:.3f}")
        print(f"  AUPRC: {best['AUPRC']:.3f}")
        print(f"  F1: {best['F1']:.3f}")
        print(f"  Precision: {best['Precision']:.3f}")
        print(f"  Recall: {best['Recall']:.3f}")
    
    print("\n" + "="*80)
    print("KEY FINDINGS")
    print("="*80)
    print("\n1. Random Split provides optimistic estimates (highest AUROC)")
    print("2. LCPO shows realistic cross-course generalization")
    print("3. Future-Presentation tests temporal generalization")
    print("4. LightGBM consistently performs best across all evaluations")
    print("5. Combined features outperform individual feature groups")
    print("6. VLE activity features are most predictive individually")
    
    print("\n" + "="*80)
    print("SAVED RESULTS")
    print("="*80)
    print(f"\n✓ Random Split: {RESULTS_DIR / 'baseline' / 'baseline_results_detailed.csv'}")
    print(f"✓ LCPO: {RESULTS_DIR / 'lcpo' / 'lcpo_results_detailed.csv'}")
    print(f"✓ Future-Presentation: {RESULTS_DIR / 'cross_course' / 'future_presentation_results.csv'}")
    print(f"✓ Comparison: {RESULTS_DIR / 'comprehensive_comparison_week16.csv'}")
    print(f"✓ Visualizations: {RESULTS_DIR / 'evaluation_comparison.png'}")
    print(f"                  {RESULTS_DIR / 'feature_group_comparison.png'}")
    
print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)